# Séance 2 — Pandas : structures et exploration (corrigé)

**Durée pratique : 3 h 30** &nbsp;·&nbsp; Ateliers 2.1 à 2.3

## Ce que vous saurez faire à la fin

- charger une même source depuis trois formats et réconcilier les types obtenus ;
- sélectionner et filtrer sans déclencher de `SettingWithCopyWarning` ;
- produire automatiquement un rapport d'exploration sur un jeu de données inconnu.

Le jeu de travail est `ventes_brutes.csv` : environ 17 600 lignes de commandes,
volontairement dégradées. Vous allez le retrouver à chaque séance jusqu'à la fin du module.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [19]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :


---
## Atelier 2.1 — Lire une source, vraiment (50 min)

### Partie guidée : la lecture naïve et ce qu'elle cache

In [20]:
sales = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')

print('Dimensions :', sales.shape)
sales.head()

FileNotFoundError: [Errno 2] No such file or directory: '/home/komlavi/Bureau/Traitement de donnees/data/raw/ventes_brutes.csv'

In [ ]:
sales.dtypes

Regardez `prix_unitaire`. Le type est `object`, autrement dit du texte, alors qu'il s'agit
d'un montant. Cherchons pourquoi.

In [ ]:
# Isoler les valeurs qui ne se convertissent pas en nombre
as_number = pd.to_numeric(sales['prix_unitaire'], errors='coerce')
unparsable = sales.loc[as_number.isna(), 'prix_unitaire']

print('Valeurs non convertibles :', len(unparsable))
print(unparsable.head(8).tolist())

Une partie des prix a été saisie avec une virgule décimale et un symbole monétaire.
Une lecture qui ignore ce détail produit une colonne texte, et tout calcul en aval échoue
silencieusement ou renvoie une erreur bien plus loin dans le pipeline.

**C'est la règle à retenir : ne jamais faire confiance à l'inférence de types.** Vérifiez
systématiquement `dtypes` après chaque lecture.

In [ ]:
def parse_price(series):
    """Convertit une colonne de prix mixte (nombre ou texte '123,45 EUR') en float."""
    text = series.astype(str).str.replace(' EUR', '', regex=False)
    text = text.str.replace(',', '.', regex=False).str.strip()
    return pd.to_numeric(text, errors='coerce')


unit_price = parse_price(sales['prix_unitaire'])
print('Valeurs encore non convertibles :', unit_price.isna().sum())
print(unit_price.describe().round(2))

### Partie autonome

In [ ]:
# Q1. Charger l'extrait des ventes depuis les trois formats disponibles,
#     puis comparer les dtypes obtenus pour la colonne 'date_commande'.
#
# Il n'existe pas de ventes_extrait.csv dans data/raw : la version CSV de la source
# est ventes_brutes.csv. On prend l'extrait CSV s'il existe, sinon le fichier brut.

csv_path = RAW_DIR / 'ventes_extrait.csv'
if not csv_path.exists():
    csv_path = RAW_DIR / 'ventes_brutes.csv'

sample_csv = pd.read_csv(csv_path)
sample_xlsx = pd.read_excel(RAW_DIR / 'ventes_extrait.xlsx')     # nécessite openpyxl
sample_json = pd.read_json(RAW_DIR / 'ventes_extrait.json')

for label, frame in [('csv', sample_csv), ('xlsx', sample_xlsx), ('json', sample_json)]:
    print(f"{label:<6} lignes={len(frame):<6} dtype date={frame['date_commande'].dtype}")

# Un aperçu des valeurs aide à comprendre d'où viennent les écarts de type
for label, frame in [('csv', sample_csv), ('xlsx', sample_xlsx), ('json', sample_json)]:
    print(f'\n{label} :', frame['date_commande'].dropna().head(3).tolist())

**Question à traiter par écrit dans la cellule suivante.** Les trois formats donnent-ils
le même type pour `date_commande` ? Le même nombre de lignes ? Que se passerait-il si votre
pipeline acceptait indifféremment ces trois sources ?

*Votre réponse :*

**Non, les trois formats ne donnent pas le même type.** Le CSV ne stocke que du texte : sans
consigne, pandas laisse `date_commande` en texte (`object`, ou `str` avec pandas 3), d'autant que
plusieurs formats de date coexistent dans la colonne. Excel stocke de vraies dates : les
cellules reconnues arrivent en `datetime64`, mais une colonne qui mélange dates et textes
retombe en `object`. Le JSON n'a pas de type date : `read_json` tente une conversion
automatique sur les colonnes dont le nom évoque une date, ce qui peut donner du
`datetime64`, des entiers (horodatages en millisecondes) ou du texte selon la façon dont le
fichier a été écrit.

**Le nombre de lignes diffère aussi** : le CSV est la source complète (≈ 17 600 lignes),
les deux autres ne sont que des extraits.

**Conséquence pour un pipeline qui accepterait ces trois sources indifféremment** : la même
information arriverait avec trois types différents. Les filtres sur les dates, les
`.dt.month` ou les concaténations échoueraient selon la source, ou pire, réussiraient en
produisant des résultats faux (dates lues comme du texte et triées par ordre alphabétique,
horodatages traités comme des nombres). La bonne pratique est d'imposer le type à la
lecture (`parse_dates`, `dtype`) puis de **vérifier `dtypes` après chaque chargement**,
quel que soit le format.

In [ ]:
# Q2. Relire ventes_brutes.csv en une seule instruction, en imposant :
#     - id_client, id_produit, id_magasin comme chaînes de caractères ;
#     - la chaîne vide et 'NC' comme valeurs manquantes ;
#     - seulement les colonnes id_commande, date_commande, id_produit, quantite,
#       prix_unitaire, statut.
#     Indice : paramètres dtype, na_values et usecols.

EXPECTED_COLUMNS = ['id_commande', 'date_commande', 'id_produit',
                    'quantite', 'prix_unitaire', 'statut']

sales_subset = pd.read_csv(
    RAW_DIR / 'ventes_brutes.csv',
    usecols=EXPECTED_COLUMNS,
    # object plutôt que str : depuis pandas 3, dtype=str crée le nouveau type 'str'
    # (StringDtype), et l'assert ci-dessous attend explicitement 'object'.
    dtype={'id_client': object, 'id_produit': object, 'id_magasin': object},
    na_values=['', 'NC'],
)[EXPECTED_COLUMNS]   # usecols ne garantit pas l'ordre : on le fixe explicitement

# Remarque : imposer dtype sur id_client et id_magasin, absentes de usecols, est sans
# effet ici. Mais c'est la bonne habitude : un identifiant comme '00123' lu en entier
# devient 123, et la jointure avec le référentiel échoue sans erreur.

assert list(sales_subset.columns) == EXPECTED_COLUMNS
assert sales_subset['id_produit'].dtype == object
print('OK —', sales_subset.shape)

---
## Atelier 2.2 — Sélection et filtrage (60 min)

### Partie guidée : `loc`, `iloc` et le piège de la copie

In [ ]:
sales['prix_unitaire'] = parse_price(sales['prix_unitaire'])
sales['montant'] = sales['quantite'] * sales['prix_unitaire']

# loc travaille sur les ÉTIQUETTES (noms de colonnes, valeurs d'index)
print(sales.loc[0:2, ['id_commande', 'quantite', 'montant']])
print()
# iloc travaille sur les POSITIONS entières
print(sales.iloc[0:2, [0, 5, -1]])

Notez la différence sur les bornes : `loc[0:2]` renvoie **trois** lignes (borne incluse),
`iloc[0:2]` en renvoie **deux** (borne exclue, comme le slicing Python).

In [ ]:
# Filtrage booléen : combiner des conditions avec & et |, chaque condition entre parenthèses
large_orders = sales[(sales['montant'] > 1000) & (sales['statut'] == 'livre')]
print('Commandes livrées de plus de 1000 EUR :', len(large_orders))

# query() est souvent plus lisible quand les conditions s'accumulent
same_result = sales.query("montant > 1000 and statut == 'livre'")
print('Même résultat :', len(same_result) == len(large_orders))

In [ ]:
# LE PIÈGE : modifier un sous-ensemble extrait par filtrage
cancelled = sales[sales['statut'] == 'annule']

# La ligne suivante déclenche un SettingWithCopyWarning : pandas ne sait pas si
# `cancelled` est une vue sur `sales` ou une copie indépendante.
cancelled['montant'] = 0

print('Montant dans sales pour les annulées :',
      sales.loc[sales['statut'] == 'annule', 'montant'].head(3).tolist())
print("-> la modification n'a PAS été propagée : le travail est perdu")

**Les deux écritures correctes**, selon l'intention :

```python
# Intention A : modifier le DataFrame d'origine
sales.loc[sales['statut'] == 'annule', 'montant'] = 0

# Intention B : travailler sur une copie indépendante
cancelled = sales[sales['statut'] == 'annule'].copy()
cancelled['montant'] = 0
```

Le message d'avertissement est le symptôme d'une ambiguïté dans votre code, pas un bruit
à faire taire.

### Partie autonome

In [ ]:
# Q3. Extraire les commandes qui remplissent TOUTES ces conditions :
#     - statut 'livre' ;
#     - quantite comprise entre 1 et 10 inclus ;
#     - montant strictement supérieur à la médiane des montants des commandes livrées ;
#     - canal contenant 'web', quelle que soit la casse.
#     Le résultat doit être une COPIE indépendante.

is_delivered = sales['statut'] == 'livre'
median_delivered = sales.loc[is_delivered, 'montant'].median()

mask = (
    is_delivered
    & sales['quantite'].between(1, 10)            # bornes incluses par défaut
    & (sales['montant'] > median_delivered)
    & sales['canal'].str.contains('web', case=False, na=False)
)
target_orders = sales.loc[mask].copy()           # .copy() : copie indépendante

print(f'Médiane des montants livrés : {median_delivered:.2f} EUR')

assert isinstance(target_orders, pd.DataFrame)
assert target_orders['quantite'].between(1, 10).all()
assert (target_orders['statut'] == 'livre').all()
assert target_orders['canal'].str.strip().str.lower().eq('web').all()
print('OK —', len(target_orders), 'commandes retenues')

In [ ]:
# Q4. Sans utiliser groupby, calculer le montant total des commandes livrées
#     pour chacun des trois canaux (après normalisation de la casse).
#     Un dictionnaire {canal: total} est attendu.

clean_channel = sales['canal'].str.strip().str.lower()
delivered = sales['statut'] == 'livre'

revenue_by_channel = {}
for channel in clean_channel.dropna().unique():
    revenue_by_channel[channel] = float(
        sales.loc[delivered & (clean_channel == channel), 'montant'].sum()
    )

assert set(revenue_by_channel) == {'web', 'boutique', 'telephone'}
print('OK')
for channel, total in sorted(revenue_by_channel.items(), key=lambda item: -item[1]):
    print(f'  {channel:<12} {total:>14,.2f} EUR'.replace(',', ' '))

---
## Atelier 2.3 — Un rapport d'exploration réutilisable (100 min)

Face à un jeu de données inconnu, les mêmes questions reviennent toujours. Plutôt que de
les reposer à la main à chaque fois, vous allez écrire une fonction qui y répond.
**Cette fonction vous servira jusqu'à la fin du module, y compris sur votre projet.**

### Partie guidée : les briques

In [ ]:
print('--- shape ---'); print(sales.shape)
print('\n--- info ---'); sales.info(memory_usage='deep')

In [ ]:
print('--- taux de valeurs manquantes ---')
missing_rate = (sales.isna().mean() * 100).round(2).sort_values(ascending=False)
print(missing_rate[missing_rate > 0])

print('\n--- cardinalité des colonnes texte ---')
for column in sales.select_dtypes(include='object').columns:
    print(f'  {column:<18} {sales[column].nunique():>6} modalités')

In [ ]:
print('--- modalités de canal, telles quelles ---')
print(sales['canal'].value_counts())

Neuf modalités pour ce qui devrait en compter trois. La casse et les espaces parasites
créent de faux niveaux. Un `value_counts()` brut est précisément l'outil qui révèle ce
genre de problème : c'est pourquoi il doit figurer dans le rapport automatique.

### Partie autonome : la fonction `profile_dataframe`

In [ ]:
def profile_dataframe(df, name='jeu de données', max_cardinality=25):
    """Affiche un rapport d'exploration standard.

    Produit, dans cet ordre :
      1. le nom, les dimensions et l'empreinte mémoire ;
      2. le nombre de lignes strictement dupliquées ;
      3. un tableau par colonne : type, nombre de valeurs manquantes, taux en %,
         nombre de valeurs distinctes ;
      4. les statistiques descriptives des colonnes numériques ;
      5. pour chaque colonne texte de cardinalité inférieure à max_cardinality,
         la répartition des modalités.

    Ne renvoie rien : la fonction affiche.
    """
    line = '=' * 72

    # 1. Identité du jeu
    memory_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    print(line)
    print(f'RAPPORT D\'EXPLORATION — {name}')
    print(line)
    print(f'Dimensions       : {df.shape[0]:,} lignes x {df.shape[1]} colonnes'.replace(',', ' '))
    print(f'Empreinte mémoire: {memory_mb:.2f} Mo')

    # 2. Doublons stricts
    n_duplicates = int(df.duplicated().sum())
    print(f'Lignes dupliquées: {n_duplicates} ({n_duplicates / max(len(df), 1):.2%})')

    # 3. Tableau par colonne
    print('\n--- Colonnes ---')
    overview = pd.DataFrame({
        'type': df.dtypes.astype(str),
        'nb_manquants': df.isna().sum(),
        'taux_manquants_pct': (df.isna().mean() * 100).round(2),
        'nb_distincts': df.nunique(dropna=True),
    })
    print(overview.to_string())

    # 4. Statistiques des colonnes numériques
    numeric = df.select_dtypes(include='number')
    print('\n--- Statistiques descriptives (numériques) ---')
    if numeric.shape[1]:
        print(numeric.describe().T.round(2).to_string())
    else:
        print('(aucune colonne numérique)')

    # 5. Modalités des colonnes texte peu cardinales
    text_columns = df.select_dtypes(include=['object', 'string', 'category']).columns
    print(f'\n--- Modalités des colonnes texte (< {max_cardinality} valeurs distinctes) ---')
    shown = 0
    for column in text_columns:
        n_unique = df[column].nunique(dropna=True)
        if n_unique < max_cardinality:
            shown += 1
            print(f'\n[{column}] {n_unique} modalités')
            print(df[column].value_counts(dropna=False).to_string())
        else:
            print(f'\n[{column}] {n_unique} modalités : trop pour être listées')
    if not len(text_columns):
        print('(aucune colonne texte)')
    print()


profile_dataframe(sales, name='ventes_brutes')

In [ ]:
# Q5. Appliquer la fonction aux deux autres jeux de données et vérifier qu'elle
#     se comporte correctement sur des structures différentes.

customers = pd.read_csv(RAW_DIR / 'clients.csv')
sensors = pd.read_csv(RAW_DIR / 'capteurs.csv')

profile_dataframe(customers, name='clients')
profile_dataframe(sensors, name='capteurs')

In [ ]:
# Q6. Déplacer la fonction dans src/exploration.py, puis l'importer ici.
#     Le notebook doit rester lisible : le code réutilisable vit dans src/.

import sys
sys.path.insert(0, str(ROOT / 'src'))

from exploration import profile_dataframe   # src/exploration.py (fourni avec le corrigé)

profile_dataframe(sales[['canal', 'statut', 'quantite']], name='test import')

---
## Ce que le rapport révèle déjà

*Vos observations :*

1. **`prix_unitaire` est lu comme du texte** : une partie des prix est saisie avec une virgule
   décimale et le suffixe ` EUR`. Aucun calcul n'est possible sans conversion explicite.
2. **`date_commande` mélange plusieurs formats** (ISO, jj/mm/aaaa, avec ou sans heure) et
   environ 2 % de valeurs manquantes. Le format jj/mm est ambigu avec mm/jj.
3. **`canal` compte neuf modalités au lieu de trois** : casse et espaces parasites créent de
   faux niveaux (`web`, `Web`, ` WEB `…). Même problème probable sur `statut` et
   `ville_livraison` (accents, casse).
4. **`remise_pct` est manquante sur environ 11 % des lignes** : à traiter explicitement
   (absence de remise ou information perdue ?).
5. **`quantite` contient des valeurs négatives** (impossibles) et des valeurs très élevées
   (plusieurs centaines) qui écrasent la moyenne et l'écart type.
6. **Doublons** : des lignes strictement identiques, et des `id_commande` répétés sur des
   lignes qui ne le sont pas.
7. **Identifiants hétérogènes** : `id_client` mélange des codes `C0xxxx` et des valeurs
   purement numériques ; il doit être lu en texte pour ne pas perdre les zéros initiaux ni
   rater les jointures.

---
## Livrable de la séance

- `src/exploration.py` contenant la fonction `profile_dataframe`, importable ;
- ce notebook exécuté, avec les six questions complétées ;
- la liste écrite des anomalies constatées.